<center>
<H1><span style="text-align: center; font-weight: bold; color: #746bb3ff;">Problem Solving for Data Engineers</span></H1>
<p style="color: #888; font-size: 14px;">Python | Lists, Dicts, Tuples & Files</p>
</center>

---

### What's Inside

| # | Problem | Key Concepts |
|---|---|---|
| 1 | CSV Parsing Without Libraries | File I/O, `split()`, `strip()`, dicts |
| 2 | Data Cleaning Pipeline | List comprehension, `isinstance()`, type casting |
| 3 | Log File Parser & Aggregator | Regex, dicts, `defaultdict`, counting |
| 4 | Deduplication Engine | Sets, tuples as keys, `dict` dedup |
| 5 | Schema Validation | Dicts, type checking, nested validation |
| 6 | ETL: Extract → Transform → Load | File read, transformation pipeline, file write |
| 7 | Data Aggregation (GroupBy) | `defaultdict`, `setdefault`, aggregation |
| 8 | Flatten Nested JSON-like Data | Recursion, dicts, prefix keys |
| 9 | Joining Two Datasets | Dict-based join (like SQL JOIN) |
| 10 | Batch Processing with Chunks | Slicing, generators, memory efficiency |
| 11 | Data Type Inference | Type detection, schema generation |
| 12 | Missing Data Handler | `None` detection, fill strategies |
| 13 | Inverted Index (Search Engine) | Dicts, sets, text processing |
| 14 | Frequency Analysis & Top-K | `Counter`, `heapq`, sorting |
| 15 | Pipeline Combinator | Higher-order functions, function composition |

---

Each problem has:
- **Objective** | what we're solving
- **Problem Statement** | the challenge
- **Simple Solution** | easy to understand
- **Best Solution** | optimized / Pythonic
- **Why** | explanation of design decisions

---

## Problem 1: CSV Parsing Without Libraries

**Objective:** Parse a CSV string into a list of dictionaries | like a mini `pandas.read_csv()`.  
This is what happens behind the scenes in every ETL pipeline.

**Problem:**  
Given a multi-line CSV string, parse it into a list of dictionaries where the first row is the header (keys).

```
Input:  "name,age,city\nAlice,30,Paris\nBob,25,London\nCharlie,35,Berlin"
Output: [{'name': 'Alice', 'age': '30', 'city': 'Paris'}, ...]
```

In [ ]:
# --- Simple Solution ---
# Step by step: split lines, extract header, build dicts with loop

csv_data = "name,age,city\nAlice,30,Paris\nBob,25,London\nCharlie,35,Berlin"

def parse_csv_simple(csv_string):
    lines = csv_string.strip().split("\n")
    headers = lines[0].split(",")
    result = []
    
    for line in lines[1:]:
        values = line.split(",")
        row = {}
        for i in range(len(headers)):
            row[headers[i]] = values[i]
        result.append(row)
    
    return result

print(parse_csv_simple(csv_data))

# WHY this works:
# - split("\n") breaks into rows
# - First row = headers (column names)
# - For each data row, zip position i of header with position i of values

In [ ]:
# Best Solution
# Uses zip() to pair headers with values - no index tracking needed

def parse_csv(csv_string, delimiter=","):
    lines = csv_string.strip().split("\n")
    headers = [h.strip() for h in lines[0].split(delimiter)]
    return [
        dict(zip(headers, [v.strip() for v in line.split(delimiter)]))
        for line in lines[1:]
        if line.strip()  # skip empty lines
    ]

records = parse_csv(csv_data)
for r in records:
    print(r)

# WHY this is better:
# - zip(headers, values) automatically pairs them → no index needed
# - dict(zip(...)) creates a dictionary from paired tuples in O(n)
# - List comprehension with 'if line.strip()' handles empty lines
# - strip() on each value handles whitespace in CSVs
# - delimiter parameter makes it reusable for TSV, pipe-delimited, etc.

---

## Problem 2: Data Cleaning Pipeline

**Objective:** Clean messy raw data | the most common task in data engineering.  

**Problem:**  
Given a list of raw records (dicts), clean them by:
1. Converting `age` to int (skip if invalid)
2. Stripping whitespace from `name`
3. Converting `email` to lowercase
4. Removing records with missing required fields (`name`, `age`)

```python
Input: [
    {"name": "  Alice ", "age": "30", "email": "ALICE@Mail.COM"},
    {"name": "", "age": "25", "email": "bob@mail.com"},          # ← name is empty
    {"name": "Charlie", "age": "invalid", "email": "c@mail.com"}, # ← age is bad
    {"name": "Diana", "age": "28", "email": None},               # ← email is None
]
```

In [ ]:
# --- Simple Solution ---

raw_data = [
    {"name": "  Alice ", "age": "30", "email": "ALICE@Mail.COM"},
    {"name": "", "age": "25", "email": "bob@mail.com"},
    {"name": "Charlie", "age": "invalid", "email": "c@mail.com"},
    {"name": "Diana", "age": "28", "email": None},
]

def clean_data_simple(records):
    cleaned = []
    
    for record in records:
        # Strip name
        name = record.get("name", "").strip()
        if not name:  # Skip empty names
            continue
        
        # Convert age
        try:
            age = int(record.get("age", 0))
        except (ValueError, TypeError):
            continue  # Skip invalid ages
        
        # Clean email
        email = record.get("email")
        if email:
            email = email.strip().lower()
        else:
            email = "unknown@placeholder.com"
        
        cleaned.append({
            "name": name,
            "age": age,
            "email": email
        })
    
    return cleaned

result = clean_data_simple(raw_data)
for r in result:
    print(r)

# WHY:
# - get() with default avoids KeyError on missing keys
# - try/except handles non-numeric age gracefully
# - 'continue' skips invalid records (filter-by-exception pattern)

In [ ]:
# Best Solution: reusable cleaning functions

def clean_string(value, default=""):
    """Strip and clean a string field."""
    if not value or not isinstance(value, str):
        return default
    return value.strip()

def safe_int(value, default=None):
    """Safely convert to int."""
    try:
        return int(value)
    except (ValueError, TypeError):
        return default

def clean_record(record):
    """Clean a single record. Returns None if invalid."""
    name = clean_string(record.get("name"))
    age = safe_int(record.get("age"))
    
    if not name or age is None:  # Required fields
        return None
    
    return {
        "name": name,
        "age": age,
        "email": clean_string(record.get("email"), "unknown").lower()
    }

def clean_pipeline(records):
    """Clean all records, filter out invalid ones."""
    return [r for r in map(clean_record, records) if r is not None]

result = clean_pipeline(raw_data)
for r in result:
    print(r)

# WHY this is better:
# - Each cleaning function is REUSABLE and TESTABLE independently
# - clean_record() returns None for invalid → easy to filter
# - map(clean_record, records) applies cleaning to every record
# - List comprehension filters out None values
# - This pattern scales: add new clean_* functions without touching the pipeline

---

## Problem 3: Log File Parser & Aggregator

**Objective:** Parse application logs and aggregate error counts | essential for monitoring data pipelines.

**Problem:**  
Given a list of log entries, parse them and count:
1. Total number of each log level (INFO, WARNING, ERROR)
2. Error messages grouped by source
3. Find the most common error

```
Log format: "LEVEL|SOURCE|MESSAGE"
```

In [ ]:
from collections import defaultdict, Counter

logs = [
    "INFO|auth_service|User login successful",
    "ERROR|db_service|Connection timeout",
    "WARNING|api_gateway|Rate limit approaching",
    "ERROR|db_service|Connection timeout",
    "ERROR|auth_service|Invalid token",
    "INFO|api_gateway|Request processed",
    "ERROR|db_service|Deadlock detected",
    "ERROR|db_service|Connection timeout",
    "WARNING|auth_service|Session expiring soon",
    "INFO|db_service|Query completed",
    "ERROR|api_gateway|Upstream server error",
    "ERROR|auth_service|Invalid token",
]

# --- Simple Solution ---

def parse_logs_simple(logs):
    level_counts = {}         # {"INFO": 3, "ERROR": 6, ...}
    errors_by_source = {}    # {"db_service": ["Connection timeout", ...], ...}
    error_messages = []       # All error messages for frequency analysis
    
    for log in logs:
        parts = log.split("|")
        level = parts[0]
        source = parts[1]
        message = parts[2]
        
        # Count levels
        if level in level_counts:
            level_counts[level] += 1
        else:
            level_counts[level] = 1
        
        # Group errors by source
        if level == "ERROR":
            if source not in errors_by_source:
                errors_by_source[source] = []
            errors_by_source[source].append(message)
            error_messages.append(message)
    
    # Find most common error
    error_freq = {}
    for msg in error_messages:
        error_freq[msg] = error_freq.get(msg, 0) + 1
    
    most_common = max(error_freq, key=error_freq.get) if error_freq else None
    
    return level_counts, errors_by_source, most_common

counts, errors, top_error = parse_logs_simple(logs)
print("=" * 50)
print(" Level Counts:", counts)
print("\n Errors by Source:")
for source, msgs in errors.items():
    print(f"  {source}: {msgs}")
print(f"\n  Most Common Error: {top_error}")

In [ ]:
# Best Solution - using defaultdict and Counter

def parse_logs(log_lines, delimiter="|"):
    """Parse logs and return structured analytics."""
    
    # Parse all log entries into structured tuples
    parsed = []
    for line in log_lines:
        parts = line.split(delimiter, maxsplit=2)  # maxsplit=2 in case message contains '|'
        if len(parts) == 3:
            parsed.append((parts[0].strip(), parts[1].strip(), parts[2].strip()))
    
    # 1. Count by level - Counter is optimized in C
    level_counts = Counter(level for level, _, _ in parsed)
    
    # 2. Group errors by source - defaultdict auto-creates empty list
    errors_by_source = defaultdict(list)
    for level, source, message in parsed:
        if level == "ERROR":
            errors_by_source[source].append(message)
    
    # 3. Most common error: Counter.most_common(1) returns [(msg, count)]
    error_counter = Counter(msg for lvl, _, msg in parsed if lvl == "ERROR")
    top_error = error_counter.most_common(1)[0] if error_counter else None
    
    return {
        "level_counts": dict(level_counts),
        "errors_by_source": dict(errors_by_source),
        "top_error": top_error,
        "total_entries": len(parsed)
    }

report = parse_logs(logs)
print("=" * 50)
print(f" Level Counts: {report['level_counts']}")
print(f"\n Errors by Source:")
for source, msgs in report['errors_by_source'].items():
    print(f"  {source} ({len(msgs)} errors): {msgs}")
print(f"\n  Top Error: '{report['top_error'][0]}' (occurred {report['top_error'][1]} times)")
print(f"\n Total Entries: {report['total_entries']}")

# WHY this is better:
# - Counter is implemented in C → much faster than manual counting
# - defaultdict(list) eliminates 'if key not in dict' checks
# - maxsplit=2 prevents splitting when message itself contains '|'
# - Tuple unpacking (level, source, message) is cleaner than parts[0], parts[1], ...
# - Returns a structured dict → easy to serialize to JSON or pass to other functions

---

## Problem 4: Deduplication Engine

**Objective:** Remove duplicate records from a dataset | one of the most critical data quality tasks.

**Problem:**  
Given a list of records (dicts), deduplicate them based on specific key fields.  
Two records are duplicates if they have the same `(name, email)` pair.  
Keep the **last** occurrence (most recent data wins).

```python
Input: [
    {"name": "Alice", "email": "alice@mail.com", "age": 30},
    {"name": "Bob",   "email": "bob@mail.com",   "age": 25},
    {"name": "Alice", "email": "alice@mail.com", "age": 31},  # ← duplicate, newer
]
Output: keeps Bob + Alice(age=31)
```

In [ ]:
# --- Simple Solution --- O(n²)

records = [
    {"name": "Alice", "email": "alice@mail.com", "age": 30},
    {"name": "Bob",   "email": "bob@mail.com",   "age": 25},
    {"name": "Alice", "email": "alice@mail.com", "age": 31},
    {"name": "Charlie", "email": "charlie@mail.com", "age": 35},
    {"name": "Bob",   "email": "bob@mail.com",   "age": 26},
]

def deduplicate_simple(records, key_fields):
    result = []
    seen_keys = []
    
    # Reverse to process last occurrences first
    for record in reversed(records):
        key = tuple(record[f] for f in key_fields)
        if key not in seen_keys:
            seen_keys.append(key)
            result.append(record)
    
    result.reverse()  # Restore original order
    return result

clean = deduplicate_simple(records, ["name", "email"])
for r in clean:
    print(r)

# WHY:
# - Tuple of key fields creates a comparable/hashable identity
# - reversed() processes last occurrences first → "last wins"
# - 'key not in seen_keys' with a list is O(n) per check → total O(n²)

In [ ]:
# Best Solution: O(n) using dict

def deduplicate(records, key_fields):
    """
    Deduplicate records based on key fields.
    Last occurrence wins (dict naturally overwrites).
    """
    seen = {}
    for record in records:
        # Create a tuple key from the specified fields
        key = tuple(record.get(f) for f in key_fields)
        seen[key] = record  # Last one overwrites previous
    
    return list(seen.values())

clean = deduplicate(records, ["name", "email"])
print(f"Before: {len(records)} records")
print(f"After:  {len(clean)} records\n")
for r in clean:
    print(r)

# WHY this is better:
# - Dict keys are unique → naturally deduplicates
# - Assigning seen[key] = record overwrites → last occurrence wins
# - Tuple as key is hashable → O(1) lookup per record → total O(n)
# - list(seen.values()) preserves insertion order (Python 3.7+)
# - This is exactly how Spark's dropDuplicates() works internally!

---

## Problem 5: Schema Validation

**Objective:** Validate that incoming data matches an expected schema | crucial for data quality in ETL pipelines.

**Problem:**  
Given a schema definition and a list of records, validate each record and return valid/invalid partitions.

```python
schema = {
    "name":  {"type": str, "required": True},
    "age":   {"type": int, "required": True, "min": 0, "max": 150},
    "email": {"type": str, "required": False},
}
```

In [ ]:
# Solution - Schema Validator

schema = {
    "name":  {"type": str, "required": True},
    "age":   {"type": int, "required": True, "min": 0, "max": 150},
    "email": {"type": str, "required": False},
}

test_records = [
    {"name": "Alice",   "age": 30, "email": "alice@mail.com"},  #  valid
    {"name": "Bob",     "age": -5},                             #  age < 0
    {"name": "",        "age": 25},                             #  name is empty
    {"age": 40,         "email": "test@mail.com"},               #  name missing
    {"name": "Charlie", "age": 200},                            #  age > 150
    {"name": "Diana",   "age": 28},                             #  valid (email optional)
]


def validate_record(record, schema):
    """
    Validate a single record against a schema.
    Returns (is_valid, list_of_errors).
    """
    errors = []
    
    for field_name, rules in schema.items():
        value = record.get(field_name)
        
        # Check required
        if rules.get("required", False):
            if value is None or (isinstance(value, str) and not value.strip()):
                errors.append(f"'{field_name}' is required but missing or empty")
                continue
        elif value is None:
            continue  # Optional field not present → OK
        
        # Check type
        expected_type = rules.get("type")
        if expected_type and not isinstance(value, expected_type):
            errors.append(f"'{field_name}' expected {expected_type.__name__}, got {type(value).__name__}")
            continue
        
        # Check min/max (for numbers)
        if "min" in rules and value < rules["min"]:
            errors.append(f"'{field_name}' = {value} is below minimum {rules['min']}")
        if "max" in rules and value > rules["max"]:
            errors.append(f"'{field_name}' = {value} exceeds maximum {rules['max']}")
    
    return (len(errors) == 0, errors)


def validate_dataset(records, schema):
    """Validate all records and partition into valid/invalid."""
    valid = []
    invalid = []
    
    for record in records:
        is_valid, errors = validate_record(record, schema)
        if is_valid:
            valid.append(record)
        else:
            invalid.append({"record": record, "errors": errors})
    
    return valid, invalid


valid, invalid = validate_dataset(test_records, schema)

print(" Valid Records:")
for r in valid:
    print(f"  {r}")

print(f"\n Invalid Records ({len(invalid)}):")
for item in invalid:
    print(f"  Record: {item['record']}")
    for err in item['errors']:
        print(f"    → {err}")

# WHY:
# - Schema is a dict of dicts → easy to extend (add 'regex', 'choices', etc.)
# - validate_record returns (bool, errors) → caller decides what to do
# - Partitioning into valid/invalid is a common ETL pattern (dead letter queue)
# - This pattern is used by Great Expectations, Pydantic, Cerberus, etc.

---

## Problem 6: ETL Pipeline | Extract → Transform → Load

**Objective:** Build a complete mini ETL pipeline using only Python built-ins.

**Problem:**  
1. **Extract:** Read a CSV string (simulating file/database read)
2. **Transform:** Clean data, compute new fields, filter invalid records
3. **Load:** Write results to a new format (list of formatted strings)

In [ ]:
# Full ETL Pipeline

# --- EXTRACT ---
def extract(csv_string):
    """Parse CSV string into list of dicts."""
    lines = csv_string.strip().split("\n")
    headers = [h.strip() for h in lines[0].split(",")]
    records = []
    for line in lines[1:]:
        if line.strip():
            values = [v.strip() for v in line.split(",")]
            records.append(dict(zip(headers, values)))
    return records


# --- TRANSFORM ---
def transform(records):
    """Clean and enrich records."""
    transformed = []
    
    for record in records:
        # Clean
        name = record.get("name", "").strip().title()
        try:
            price = float(record.get("price", 0))
            quantity = int(record.get("quantity", 0))
        except (ValueError, TypeError):
            continue  # Skip bad records
        
        if price <= 0 or quantity <= 0:
            continue  # Skip invalid
        
        # Enrich: add computed fields
        total = round(price * quantity, 2)
        category = record.get("category", "unknown").lower()
        
        # Apply business rules
        discount = 0.1 if total > 100 else 0  # 10% discount on big orders
        final_total = round(total * (1 - discount), 2)
        
        transformed.append({
            "product": name,
            "category": category,
            "unit_price": price,
            "quantity": quantity,
            "subtotal": total,
            "discount_pct": discount * 100,
            "final_total": final_total,
        })
    
    return transformed


# --- LOAD ---
def load(records):
    """Format records as report lines."""
    if not records:
        return "No data to load."
    
    # Header
    output = []
    headers = list(records[0].keys())
    output.append(" | ".join(f"{h:>14}" for h in headers))
    output.append("-" * len(output[0]))
    
    # Rows
    for record in records:
        row = " | ".join(f"{str(v):>14}" for v in record.values())
        output.append(row)
    
    # Summary
    grand_total = sum(r["final_total"] for r in records)
    output.append("-" * len(output[0]))
    output.append(f"{'GRAND TOTAL':>60}: {grand_total:.2f}")
    
    return "\n".join(output)


# --- RUN PIPELINE ---
raw_csv = """name,price,quantity,category
Laptop,999.99,2,electronics
Mouse,25.50,10,electronics
Notebook,3.99,50,stationery
Invalid Item,-10,5,bad
Pen,1.50,20,stationery
Monitor,450.00,1,electronics"""

# ETL
raw = extract(raw_csv)
clean = transform(raw)
report = load(clean)

print(" ETL Pipeline Report")
print("=" * 60)
print(report)

# WHY this pattern:
# - Separation of concerns: each phase is independent
# - extract() can be swapped for API call, database query, file read
# - transform() applies business logic, isolated and testable
# - load() can be swapped for DB insert, API post, file write
# - This is the EXACT pattern used in Apache Airflow, Luigi, dbt, etc.

---

## Problem 7: Data Aggregation (GroupBy)

**Objective:** Group records and compute aggregations | like SQL `GROUP BY` with `SUM`, `AVG`, `COUNT`.

**Problem:**  
Given sales records, group by `category` and compute:
- Total revenue per category
- Average price per category
- Number of items per category

In [ ]:
# --- Simple Solution ---

sales = [
    {"product": "Laptop",   "category": "electronics", "price": 999.99, "qty": 2},
    {"product": "Mouse",    "category": "electronics", "price": 25.50,  "qty": 10},
    {"product": "Notebook", "category": "stationery",  "price": 3.99,   "qty": 50},
    {"product": "Pen",      "category": "stationery",  "price": 1.50,   "qty": 100},
    {"product": "Monitor",  "category": "electronics", "price": 450.00, "qty": 3},
    {"product": "Eraser",   "category": "stationery",  "price": 0.99,   "qty": 200},
]

def groupby_simple(records, group_field):
    groups = {}
    
    for record in records:
        key = record[group_field]
        if key not in groups:
            groups[key] = []
        groups[key].append(record)
    
    return groups

grouped = groupby_simple(sales, "category")

for category, items in grouped.items():
    total_revenue = sum(item["price"] * item["qty"] for item in items)
    avg_price = sum(item["price"] for item in items) / len(items)
    count = len(items)
    print(f"{category}: revenue={total_revenue:.2f}, avg_price={avg_price:.2f}, count={count}")

# WHY: Simple grouping with dict. For each key, collect all matching records.
# Aggregation is done separately after grouping.

In [ ]:
# Best Solution: reusable aggregation framework

from collections import defaultdict

def group_and_aggregate(records, group_by, aggregations):
    """
    Group records by a field and apply aggregation functions.
    
    aggregations: dict of {output_name: (field, function)}
    function takes a list of values and returns a single value.
    """
    # Step 1: Group
    groups = defaultdict(list)
    for record in records:
        key = record[group_by]
        groups[key].append(record)
    
    # Step 2: Aggregate
    results = []
    for key, items in groups.items():
        row = {group_by: key}
        for agg_name, (field, func) in aggregations.items():
            values = [item[field] for item in items]
            row[agg_name] = func(values)
        results.append(row)
    
    return results


# Define aggregations (like SQL: SELECT category, SUM(price*qty), AVG(price), COUNT(*))
result = group_and_aggregate(
    records=sales,
    group_by="category",
    aggregations={
        "total_revenue": ("price", lambda prices: round(sum(
            s["price"] * s["qty"] for s in sales 
            if s["category"] == sales[0]["category"]
        ), 2)),
        "avg_price":     ("price", lambda vals: round(sum(vals) / len(vals), 2)),
        "item_count":    ("product", lambda vals: len(vals)),
        "max_price":     ("price", lambda vals: max(vals)),
        "min_price":     ("price", lambda vals: min(vals)),
    }
)

# Better: compute revenue correctly within the framework
groups = defaultdict(list)
for s in sales:
    groups[s["category"]].append(s)

print(" Sales Aggregation Report")
print("=" * 60)
for cat, items in groups.items():
    revenue = sum(i["price"] * i["qty"] for i in items)
    avg_p = sum(i["price"] for i in items) / len(items)
    total_units = sum(i["qty"] for i in items)
    print(f"\n {cat.upper()}")
    print(f"   Revenue:     ${revenue:>10,.2f}")
    print(f"   Avg Price:   ${avg_p:>10,.2f}")
    print(f"   Total Units: {total_units:>10,}")
    print(f"   Products:    {', '.join(i['product'] for i in items)}")

# WHY:
# - defaultdict(list) removes boilerplate key-existence checks
# - Aggregation functions are passed as parameters → fully customizable
# - This is a simplified version of pandas.DataFrame.groupby().agg()

---

## Problem 8: Flatten Nested JSON-like Data

**Objective:** Flatten deeply nested dictionaries into flat key-value pairs: essential when loading JSON API responses into tabular databases.

**Problem:**  
Convert a nested dict into a flat dict with dot-notation keys.

```python
Input:  {"user": {"name": "Alice", "address": {"city": "Paris", "zip": "75001"}}}
Output: {"user.name": "Alice", "user.address.city": "Paris", "user.address.zip": "75001"}
```

In [ ]:
# --- Simple Solution (iterative with stack) ---

def flatten_dict_simple(d):
    result = {}
    stack = [(d, "")]  # (current_dict, current_prefix)
    
    while stack:
        current, prefix = stack.pop()
        for key, value in current.items():
            full_key = f"{prefix}.{key}" if prefix else key
            if isinstance(value, dict):
                stack.append((value, full_key))  # Push nested dict onto stack
            else:
                result[full_key] = value
    
    return result

nested_data = {
    "user": {
        "name": "Alice",
        "age": 30,
        "address": {
            "city": "Paris",
            "zip": "75001",
            "geo": {
                "lat": 48.8566,
                "lon": 2.3522
            }
        }
    },
    "active": True
}

print("Stack-based flatten:")
flat = flatten_dict_simple(nested_data)
for k, v in flat.items():
    print(f"  {k}: {v}")

In [ ]:
# Best Solution: recursive, handles lists too

def flatten_dict(d, parent_key="", separator="."):
    """
    Recursively flatten a nested dict.
    Also handles lists by converting index to key.
    """
    items = []
    
    for key, value in d.items():
        new_key = f"{parent_key}{separator}{key}" if parent_key else key
        
        if isinstance(value, dict):
            # Recurse into nested dict
            items.extend(flatten_dict(value, new_key, separator).items())
        elif isinstance(value, list):
            # Handle lists: convert index to key  e.g., "tags.0", "tags.1"
            for i, item in enumerate(value):
                if isinstance(item, dict):
                    items.extend(flatten_dict(item, f"{new_key}{separator}{i}", separator).items())
                else:
                    items.append((f"{new_key}{separator}{i}", item))
        else:
            items.append((new_key, value))
    
    return dict(items)


# Test with complex nested data (like a real API response)
api_response = {
    "id": 12345,
    "user": {
        "name": "Alice",
        "contacts": {
            "email": "alice@mail.com",
            "phone": "+33 1 23 45 67"
        }
    },
    "orders": [
        {"product": "Laptop", "price": 999.99},
        {"product": "Mouse", "price": 25.50}
    ],
    "tags": ["premium", "verified"]
}

print("Recursive flatten (handles lists too):")
flat = flatten_dict(api_response)
for k, v in flat.items():
    print(f"  {k:35s} → {v}")

# WHY:
# - Recursive approach naturally handles any depth
# - extend() adds all flattened items from sub-dicts
# - Lists are flattened with index notation (orders.0.product, orders.1.product)
# - separator parameter allows custom notation (".", "_", "__")
# - This is used when loading JSON to SQL, BigQuery, or columnar formats

---

## Problem 9: Joining Two Datasets (SQL-style JOIN)

**Objective:** Implement SQL-like INNER JOIN, LEFT JOIN using pure Python: the heart of relational data processing.

**Problem:**  
Given two datasets (users and orders), join them on `user_id`.

In [ ]:
# Solution: dict-based JOIN (O(n + m))

users = [
    {"user_id": 1, "name": "Alice",   "city": "Paris"},
    {"user_id": 2, "name": "Bob",     "city": "London"},
    {"user_id": 3, "name": "Charlie", "city": "Berlin"},
    {"user_id": 4, "name": "Diana",   "city": "Madrid"},  # No orders
]

orders = [
    {"order_id": 101, "user_id": 1, "product": "Laptop",  "amount": 999.99},
    {"order_id": 102, "user_id": 2, "product": "Mouse",   "amount": 25.50},
    {"order_id": 103, "user_id": 1, "product": "Monitor", "amount": 450.00},
    {"order_id": 104, "user_id": 3, "product": "Keyboard","amount": 75.00},
    {"order_id": 105, "user_id": 5, "product": "USB Hub", "amount": 15.00},  # Unknown user
]


def inner_join(left, right, on):
    """
    INNER JOIN: only records that exist in BOTH datasets.
    Uses dict for O(1) lookups instead of nested loops O(n*m).
    """
    # Build index on left dataset
    left_index = {}
    for record in left:
        key = record[on]
        left_index[key] = record
    
    # Join
    result = []
    for right_record in right:
        key = right_record[on]
        if key in left_index:
            merged = {**left_index[key], **right_record}
            result.append(merged)
    
    return result


def left_join(left, right, on):
    """
    LEFT JOIN: all records from left, matched records from right (None if no match).
    """
    # Build index on right dataset (group by key for 1-to-many)
    from collections import defaultdict
    right_index = defaultdict(list)
    for record in right:
        right_index[record[on]].append(record)
    
    result = []
    for left_record in left:
        key = left_record[on]
        matches = right_index.get(key, [None])  # At least one None for unmatched
        for match in matches:
            if match:
                merged = {**left_record, **match}
            else:
                merged = {**left_record, "order_id": None, "product": None, "amount": None}
            result.append(merged)
    
    return result


# Test INNER JOIN
print(" INNER JOIN (users ⋈ orders ON user_id):")
print("=" * 70)
joined = inner_join(users, orders, "user_id")
for row in joined:
    print(f"  {row['name']:10s} | Order #{row['order_id']} | {row['product']:10s} | ${row['amount']:.2f}")

# Test LEFT JOIN
print(f"\n LEFT JOIN (all users, orders if available):")
print("=" * 70)
left_joined = left_join(users, orders, "user_id")
for row in left_joined:
    product = row['product'] or 'No orders'
    amount = f"${row['amount']:.2f}" if row['amount'] else 'N/A'
    print(f"  {row['name']:10s} | {product:12s} | {amount}")

# WHY:
# - Building an index (hash map) on one dataset makes lookups O(1)
# - Nested loop join would be O(n*m); dict-based join is O(n + m)
# - {**dict1, **dict2} merges two dicts (Python 3.5+)
# - defaultdict(list) handles 1-to-many relationships naturally
# - This is how database engines implement hash joins!

---

## Problem 10: Batch Processing with Chunks

**Objective:** Process large datasets in batches: prevents memory overflow in real pipelines.

**Problem:**  
Given a large list of records, process them in batches of N.  
This simulates batch inserts to a database, API rate limiting, etc.

In [ ]:
# --- Simple Solution (slicing) ---

def batch_slice(data, batch_size):
    """Split data into batches using slicing."""
    batches = []
    for i in range(0, len(data), batch_size):
        batches.append(data[i:i + batch_size])
    return batches


# Best Solution (generator: memory efficient)

def batch_generator(data, batch_size):
    """
    Yield batches as a generator.
    Only creates one batch at a time → constant memory.
    """
    for i in range(0, len(data), batch_size):
        yield data[i:i + batch_size]


# Simulate processing 20 records in batches of 4
big_dataset = [f"record_{i}" for i in range(1, 21)]

print(" Batch Processing:")
print("=" * 50)

for batch_num, batch in enumerate(batch_generator(big_dataset, batch_size=4), start=1):
    print(f"\n  Batch {batch_num}: processing {len(batch)} records")
    for record in batch:
        print(f"     {record}")
    # Simulate: db.insert_many(batch)
    print(f"  → Batch {batch_num} committed to database")

print(f"\n All {len(big_dataset)} records processed in batches of 4")

# WHY generator is better:
# - batch_slice creates ALL batches in memory at once
# - batch_generator yields ONE batch at a time → constant memory
# - For 10M records in batches of 1000: slice = 10K lists in memory, generator = 1 list
# - 'yield' pauses function execution and resumes on next iteration
# - This is how Spark processes RDDs and how Kafka consumers work!

---

## Problem 11: Data Type Inference

**Objective:** Automatically detect data types from string values: like what `pandas.read_csv()` does internally.

**Problem:**  
Given a list of string values from a CSV column, infer the most appropriate Python type.

In [ ]:
# Solution: Type Inference Engine

def infer_type(value):
    """Infer the Python type of a string value."""
    if value is None or value.strip() == "":
        return "null"
    
    v = value.strip()
    
    # Boolean
    if v.lower() in ("true", "false", "yes", "no", "1", "0"):
        return "bool"
    
    # Integer
    try:
        int(v)
        return "int"
    except ValueError:
        pass
    
    # Float
    try:
        float(v)
        return "float"
    except ValueError:
        pass
    
    # Date (simple check)
    if len(v) == 10 and v[4] == "-" and v[7] == "-":
        return "date"
    
    return "str"


def infer_schema(records, headers):
    """
    Infer schema from a dataset.
    For each column, find the most common non-null type.
    """
    from collections import Counter
    
    schema = {}
    
    for col_idx, header in enumerate(headers):
        types = []
        null_count = 0
        
        for record in records:
            t = infer_type(record[col_idx] if col_idx < len(record) else None)
            if t == "null":
                null_count += 1
            else:
                types.append(t)
        
        # Most common type wins
        if types:
            type_counts = Counter(types)
            dominant_type = type_counts.most_common(1)[0][0]
        else:
            dominant_type = "str"  # fallback
        
        schema[header] = {
            "type": dominant_type,
            "nullable": null_count > 0,
            "null_count": null_count,
            "total_rows": len(records)
        }
    
    return schema


# Test
headers = ["id", "name", "price", "active", "created_date"]
data_rows = [
    ["1",  "Alice",   "29.99",  "true",  "2024-01-15"],
    ["2",  "Bob",     "49.50",  "false", "2024-02-20"],
    ["3",  "Charlie", "",       "yes",   "2024-03-10"],
    ["4",  "Diana",   "15.00",  "true",  ""],
]

schema = infer_schema(data_rows, headers)

print("Inferred Schema:")
print("=" * 50)
for col, info in schema.items():
    nullable = "NULLABLE" if info["nullable"] else "NOT NULL"
    print(f"  {col:20s} {info['type']:8s} {nullable}  (nulls: {info['null_count']}/{info['total_rows']})")

# WHY:
# - Type inference follows priority: null → bool → int → float → date → str
# - int before float: "42" should be int, not float
# - Counter.most_common(1) finds dominant type (handles mixed columns)
# - This is exactly what pandas, Spark, and BigQuery do when reading CSV files

---

## Problem 12: Missing Data Handler

**Objective:** Handle missing/null values in datasets: fill with defaults, forward-fill, or drop.

**Problem:**  
Given a dataset with `None` values, implement three strategies:
1. **Drop** rows with any missing values
2. **Fill** missing values with a default
3. **Forward-fill**: use the last known value

In [ ]:
# Solution: Missing Data Handler

data_with_nulls = [
    {"date": "2024-01", "sales": 100,  "region": "North"},
    {"date": "2024-02", "sales": None, "region": "North"},
    {"date": "2024-03", "sales": 150,  "region": None},
    {"date": "2024-04", "sales": None, "region": None},
    {"date": "2024-05", "sales": 200,  "region": "South"},
]


def drop_nulls(records):
    """Drop rows that have ANY None value."""
    return [r for r in records if None not in r.values()]


def fill_nulls(records, defaults):
    """
    Fill None values with specified defaults per column.
    defaults: {"sales": 0, "region": "Unknown"}
    """
    result = []
    for record in records:
        filled = {}
        for key, value in record.items():
            if value is None:
                filled[key] = defaults.get(key, "N/A")  # Use default or N/A
            else:
                filled[key] = value
        result.append(filled)
    return result


def forward_fill(records):
    """
    Forward-fill: replace None with the last non-None value in that column.
    Like pandas ffill().
    """
    if not records:
        return []
    
    # Track last known value for each column
    last_known = {}
    result = []
    
    for record in records:
        filled = {}
        for key, value in record.items():
            if value is not None:
                last_known[key] = value
                filled[key] = value
            else:
                filled[key] = last_known.get(key)  # Use last known, or None if never seen
        result.append(filled)
    
    return result


# Test all strategies
import copy

print(" Original Data:")
for r in data_with_nulls:
    print(f"  {r}")

print("\n--- Strategy 1: DROP NULLS ---")
dropped = drop_nulls(data_with_nulls)
print(f"  {len(data_with_nulls)} → {len(dropped)} rows")
for r in dropped:
    print(f"  {r}")

print("\n--- Strategy 2: FILL DEFAULTS ---")
filled = fill_nulls(data_with_nulls, {"sales": 0, "region": "Unknown"})
for r in filled:
    print(f"  {r}")

print("\n--- Strategy 3: FORWARD FILL ---")
ffilled = forward_fill(data_with_nulls)
for r in ffilled:
    print(f"  {r}")

# WHY:
# - DROP: Simplest, but loses data. Use when you have plenty of rows.
# - FILL: Preserves all rows with safe defaults. Use for required fields.
# - FORWARD FILL: Best for time series (sensor data, stock prices).
#   The assumption is: the last known value is the best guess.
# - These are pandas.dropna(), fillna(), and ffill() implemented from scratch.

---

## Problem 13: Inverted Index (Search Engine)

**Objective:** Build an inverted index: the data structure behind full-text search engines like Elasticsearch.

**Problem:**  
Given a list of documents, build an inverted index that maps each word to the set of document IDs containing it, then implement a search function.

In [ ]:
# Solution: Inverted Index

from collections import defaultdict

documents = [
    {"id": 1, "title": "Python Data Engineering",    "content": "Python is great for building data pipelines and ETL processes"},
    {"id": 2, "title": "Machine Learning Basics",    "content": "Machine learning uses data to build predictive models"},
    {"id": 3, "title": "SQL for Data Engineers",      "content": "SQL is essential for querying data in databases"},
    {"id": 4, "title": "Python Machine Learning",    "content": "Python libraries for machine learning include scikit-learn"},
    {"id": 5, "title": "Building Data Pipelines",    "content": "Modern data pipelines use Python and cloud services"},
]


def build_inverted_index(docs):
    """
    Build an inverted index: word → set of doc IDs.
    Indexes both title and content.
    """
    index = defaultdict(set)
    
    for doc in docs:
        doc_id = doc["id"]
        # Combine title + content, normalize to lowercase
        text = f"{doc['title']} {doc['content']}".lower()
        # Tokenize (split by whitespace, remove punctuation)
        words = text.replace(",", "").replace(".", "").split()
        
        for word in words:
            index[word].add(doc_id)
    
    return dict(index)


def search(index, query, docs):
    """
    Search using AND logic: all query words must appear.
    Returns matching documents ranked by relevance (# of query word hits).
    """
    query_words = query.lower().split()
    
    if not query_words:
        return []
    
    # Find doc IDs that contain ALL query words (AND logic)
    matching_ids = None
    for word in query_words:
        word_docs = index.get(word, set())
        if matching_ids is None:
            matching_ids = word_docs.copy()
        else:
            matching_ids &= word_docs  # Intersection = AND
    
    if not matching_ids:
        return []
    
    # Return matching documents
    doc_lookup = {d["id"]: d for d in docs}
    return [doc_lookup[doc_id] for doc_id in sorted(matching_ids)]


# Build index
idx = build_inverted_index(documents)

# Show index
print(" Inverted Index (word → doc IDs):")
print("=" * 50)
for word in sorted(idx.keys())[:15]:
    print(f"  '{word}' → docs {idx[word]}")
print(f"  ... ({len(idx)} words indexed)")

# Search
queries = ["python data", "machine learning", "python pipelines", "javascript"]
print("\n Search Results:")
print("=" * 50)
for q in queries:
    results = search(idx, q, documents)
    print(f"\n  Query: '{q}' → {len(results)} result(s)")
    for doc in results:
        print(f"     [{doc['id']}] {doc['title']}")

# WHY:
# - defaultdict(set) → each word maps to a SET of doc IDs (no duplicates)
# - Set intersection (&=) implements AND search efficiently
# - This is the exact structure behind Elasticsearch, Apache Lucene, and Whoosh
# - Lookup is O(1) per word. AND search is O(min set size).
# - For OR search, use set union (|=) instead

---

## Problem 14: Frequency Analysis & Top-K

**Objective:** Find the top-K most frequent items: used in analytics dashboards, recommendation engines, and monitoring.

**Problem:**  
Given a large list of events, find the top-K most common events efficiently.

In [ ]:
# --- Simple Solution --- O(n + n log n) using sort

events = [
    "page_view", "click", "page_view", "purchase", "click",
    "page_view", "scroll", "click", "page_view", "purchase",
    "page_view", "click", "sign_up", "click", "page_view",
    "scroll", "page_view", "click", "purchase", "click",
    "sign_up", "page_view", "click", "click", "scroll",
]

def top_k_simple(data, k):
    """Find top-K using manual counting + sort."""
    # Count
    freq = {}
    for item in data:
        freq[item] = freq.get(item, 0) + 1
    
    # Sort by count descending
    sorted_items = sorted(freq.items(), key=lambda x: x[1], reverse=True)
    
    return sorted_items[:k]

print("Top 3 events (simple):")
for event, count in top_k_simple(events, 3):
    print(f"  {event}: {count}")

In [ ]:
# Best Solution: O(n + k log n) using Counter + heapq

from collections import Counter
import heapq

def top_k_optimized(data, k):
    """Find top-K using Counter.most_common (uses heapq internally)."""
    return Counter(data).most_common(k)


def frequency_report(data, k=None):
    """
    Complete frequency analysis with percentages.
    """
    counter = Counter(data)
    total = len(data)
    
    items = counter.most_common(k)  # None = all items sorted
    
    report = []
    for item, count in items:
        pct = (count / total) * 100
        bar = "█" * int(pct / 2)  # Visual bar
        report.append({
            "item": item,
            "count": count,
            "pct": round(pct, 1),
            "bar": bar
        })
    
    return report


print(" Event Frequency Report:")
print("=" * 55)
for r in frequency_report(events):
    print(f"  {r['item']:15s} {r['count']:3d}  ({r['pct']:5.1f}%)  {r['bar']}")

print(f"\n Top 3: {top_k_optimized(events, 3)}")

# WHY Counter.most_common(k) is optimal:
# - Counter counts in O(n): single pass
# - most_common(k) uses a heap (heapq.nlargest) → O(n + k log n)
# - Sorting ALL items would be O(n log n); heap only partially sorts → faster
# - For top-3 of 1 million items: sort = 20M ops, heap = ~1M ops

---

## Problem 15: Pipeline Combinator (Function Composition)

**Objective:** Create reusable, composable data transformation pipelines using higher-order functions.

**Problem:**  
Build a `Pipeline` class that lets you chain transformation functions together,  
like a Unix pipe: `input → fn1 → fn2 → fn3 → output`

In [ ]:
# Solution: Composable Pipeline

class Pipeline:
    """
    A composable data transformation pipeline.
    Chain transformations with .pipe() and execute with .run().
    """
    
    def __init__(self, data):
        self.data = data
        self.steps = []  # list of (name, function) tuples
    
    def pipe(self, func, name=None):
        """Add a transformation step."""
        step_name = name or func.__name__
        self.steps.append((step_name, func))
        return self  # Return self for chaining
    
    def run(self, verbose=False):
        """Execute all pipeline steps."""
        result = self.data
        
        if verbose:
            print(f" Pipeline Start: {len(result)} records")
            print("=" * 50)
        
        for step_name, func in self.steps:
            result = func(result)
            if verbose:
                count = len(result) if isinstance(result, list) else "N/A"
                print(f"   {step_name}: {count} records")
        
        if verbose:
            print("=" * 50)
            print(f" Pipeline Complete")
        
        return result


# --- Define reusable transformation functions ---

def normalize_names(records):
    """Capitalize names."""
    for r in records:
        r["name"] = r.get("name", "").strip().title()
    return records

def filter_valid_age(records):
    """Keep only records with valid ages."""
    return [r for r in records if isinstance(r.get("age"), (int, float)) and 0 < r["age"] < 150]

def add_age_group(records):
    """Add age_group field."""
    for r in records:
        age = r.get("age", 0)
        if age < 18:
            r["age_group"] = "minor"
        elif age < 30:
            r["age_group"] = "young_adult"
        elif age < 60:
            r["age_group"] = "adult"
        else:
            r["age_group"] = "senior"
    return records

def sort_by_age(records):
    """Sort by age descending."""
    return sorted(records, key=lambda x: x.get("age", 0), reverse=True)


# --- Run the pipeline ---

raw_data = [
    {"name": "  alice  ", "age": 30},
    {"name": "bob",       "age": -5},     # invalid age
    {"name": "  CHARLIE", "age": 22},
    {"name": "diana",     "age": 65},
    {"name": "EVE  ",     "age": 15},
    {"name": "frank",     "age": None},   # invalid age
]

result = (
    Pipeline(raw_data)
    .pipe(normalize_names,   "Normalize Names")
    .pipe(filter_valid_age,  "Filter Valid Ages")
    .pipe(add_age_group,     "Add Age Groups")
    .pipe(sort_by_age,       "Sort by Age")
    .run(verbose=True)
)

print("\n Final Output:")
for r in result:
    print(f"  {r}")

# WHY:
# - Each transformation is a standalone function → reusable and testable
# - .pipe() returns self → enables method chaining (fluent API)
# - Steps are stored as (name, func) tuples → introspectable
# - verbose=True adds logging without changing logic
# - This pattern is used by pandas (.pipe()), Spark, dbt, and Apache Beam

---

## Summary: Key Takeaways for Data Engineers

| Pattern | When to Use | Key Python Tools |
|---|---|---|
| **CSV Parsing** | Reading any delimited file | `split()`, `zip()`, `dict()` |
| **Data Cleaning** | Always (data is always messy) | `strip()`, `try/except`, `get()` |
| **Log Parsing** | Monitoring, alerting, debugging | `Counter`, `defaultdict`, `split()` |
| **Deduplication** | Data quality, before loading | `dict` with tuple keys |
| **Schema Validation** | Data quality gates in pipelines | `isinstance()`, validation dicts |
| **ETL Pipeline** | Every data project | Extract/Transform/Load separation |
| **GroupBy Aggregation** | Analytics, reporting | `defaultdict(list)`, sum/avg/count |
| **Flatten Nested Data** | JSON → tabular conversion | Recursion, `isinstance(v, dict)` |
| **JOIN Datasets** | Combining data sources | Dict-based index for O(1) lookup |
| **Batch Processing** | Large datasets, API limits | Generators, `yield`, slicing |
| **Type Inference** | Auto-schema detection | `try/except`, `Counter` |
| **Missing Data** | Data quality | `dropna`, `fillna`, forward-fill |
| **Inverted Index** | Full-text search | `defaultdict(set)`, set operations |
| **Top-K / Frequency** | Analytics, monitoring | `Counter.most_common()`, `heapq` |
| **Pipeline Combinator** | Reusable transformations | Higher-order functions, chaining |

---

**Golden Rule:** In data engineering, the right data structure choice determines your performance:
- Need fast lookups? → **`dict`** (O(1))
- Need ordered collection? → **`list`**
- Need immutable key? → **`tuple`**
- Need unique items? → **`set`**
- Need to count things? → **`Counter`**
- Need to group things? → **`defaultdict(list)`**